In [1]:
%cd /content/llm-qa-project
!git fetch origin
!git checkout prompt-engineering
!git pull

/content/llm-qa-project
branch 'prompt-engineering' set up to track 'origin/prompt-engineering'.
Switched to a new branch 'prompt-engineering'
Already up to date.


In [3]:
!grep -c "def cot" src/prompts.py

1


In [4]:
!ls results/

baseline_preds.jsonl  baseline_scores.json


In [5]:
import json
with open("results/run_log.jsonl", "a") as f:
    f.write(json.dumps({"tag": "baseline", "n_generated": 1000, "minutes": 8.7, "gpu": "Tesla T4",
                        "batch_size": 8, "max_new_tokens": 48, "time_utc": "2026-09-29"}) + "\n")
!python -m src.run_inference --method baseline

Traceback (most recent call last):
  File "<frozen runpy>", line 203, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/content/llm-qa-project/src/run_inference.py", line 134, in <module>
    main()
    ~~~~^^
  File "/content/llm-qa-project/src/run_inference.py", line 87, in main
    test = load_jsonl(args.test)
  File "/content/llm-qa-project/src/evaluate_qa.py", line 136, in load_jsonl
    with open(path, encoding="utf-8") as f:
         ~~~~^^^^^^^^^^^^^^^^^^^^^^^^
FileNotFoundError: [Errno 2] No such file or directory: 'data/test.jsonl'


In [6]:
!python -m src.data_prep --test_size 1000

README.md: 100% 8.92k/8.92k [00:00<00:00, 18.8MB/s]

squad_v2/train-00000-of-00001.parquet: downloading bytes:  73% 12.0M/16.4M [00:00<00:00, 22.8MB/s,  466kB/s  ]
squad_v2/train-00000-of-00001.parquet: downloading bytes: 100% 15.9M/15.9M [00:00<00:00, 23.1MB/s, 1.56MB/s  ]
squad_v2/train-00000-of-00001.parquet: reconstructing file: 100% 16.4M/16.4M [00:00<00:00, 23.8MB/s, 1.61MB/s  ]

squad_v2/validation-00000-of-00001.parqu(…): downloading bytes:   0% 0.00/1.35M [00:00<?, ?B/s]s]
squad_v2/validation-00000-of-00001.parqu(…): downloading bytes: 100% 1.30M/1.30M [00:00<00:00, 2.35MB/s,  128kB/s  ]
squad_v2/validation-00000-of-00001.parqu(…): reconstructing file: 100% 1.35M/1.35M [00:00<00:00, 2.45MB/s,  133kB/s  ]
Generating train split: 100% 130319/130319 [00:00<00:00, 470963.98 examples/s]
Generating validation split: 100% 11873/11873 [00:00<00:00, 260795.87 examples/s]
wrote   1000 rows -> data/test.jsonl
wrote 130319 rows -> data/train.jsonl
wrote     20 rows -> data/fewshot_pool.js

In [7]:
!ls results/

baseline_preds.jsonl  baseline_scores.json  run_log.jsonl


In [8]:
!python -m src.run_inference --method baseline

Resuming: 1000 of 1000 already done
{
  "n_total": 1000,
  "n_answerable": 493,
  "n_unanswerable": 507,
  "exact_match": 1.9,
  "f1": 17.51,
  "lenient_accuracy": 45.1,
  "has_ans_em": 0.2,
  "has_ans_f1": 31.87,
  "no_ans_accuracy": 3.55,
  "bleu": 11.99,
  "rouge_l": 31.23,
  "method": "baseline",
  "model": "microsoft/Phi-3-mini-4k-instruct",
  "generation_minutes": 8.7,
  "gpu": "Tesla T4",
  "max_new_tokens": 48
}

Saved results/baseline_preds.jsonl and results/baseline_scores.json


In [9]:
!python -m src.run_inference --method fewshot3 --limit 20

Loading microsoft/Phi-3-mini-4k-instruct ...
config.json: 100% 967/967 [00:00<00:00, 3.07MB/s]
tokenizer_config.json: 100% 3.44k/3.44k [00:00<00:00, 5.41MB/s]
tokenizer.json: 100% 1.94M/1.94M [00:00<00:00, 72.2MB/s]

tokenizer.model: downloading bytes:  17% 86.7k/500k [00:00<00:02, 166kB/s]
tokenizer.model: downloading bytes: 100% 346k/346k [00:00<00:00, 658kB/s, 34.2kB/s  ]
tokenizer.model: reconstructing file: 100% 500k/500k [00:00<00:00, 949kB/s, 49.4kB/s  ]
added_tokens.json: 100% 306/306 [00:00<00:00, 1.36MB/s]
special_tokens_map.json: 100% 599/599 [00:00<00:00, 3.75MB/s]
model.safetensors.index.json: 100% 16.5k/16.5k [00:00<00:00, 42.9MB/s]
Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0% 0/2 [00:00<?, ?it/s]
Reconstructing (incomplete total...):   0% 0.00/2.67G [00:00<?, ?B/s]         
Reconstructing (incomplete total...):   3% 201M/7.64G [00:01<01:04, 116MB/s]
Reconstructing (incomplete total...):  34% 2.57G/7.64G [00:17<00:

In [10]:
import json
print(json.dumps(json.load(open("results/fewshot3_sample20_scores.json")), indent=2))
for r in map(json.loads, open("results/fewshot3_sample20_preds.jsonl")):
    print(repr(r["prediction"][:80]))

{
  "n_total": 20,
  "n_answerable": 8,
  "n_unanswerable": 12,
  "exact_match": 25.0,
  "f1": 41.05,
  "lenient_accuracy": 50.0,
  "has_ans_em": 37.5,
  "has_ans_f1": 77.63,
  "no_ans_accuracy": 16.67,
  "bleu": 60.4,
  "rouge_l": 76.28,
  "method": "fewshot3",
  "model": "microsoft/Phi-3-mini-4k-instruct",
  "generation_minutes": 0.2,
  "gpu": "Tesla T4",
  "max_new_tokens": 48
}
'MLB (Major League Baseball)'
'Los Angeles Kings, Anaheim Ducks'
'Pacific Ocean'
'unanswerable'
'unanswerable'
'applied force'
'Polish and international artists'
'2000'
'land and housing'
'2p − 1, with p a prime'
'higher'
'The Mask of Anarchy'
'a plug valve in the top of a boiler.'
'green spaces'
'The accidental introduction of Beroe helped to mitigate the problem.'
'Varsovians thought of themselves as better only because they lived in the capita'
'weathering and erosion'
'two main classes'
'Fort Presque Isle (near present-day Erie, Pennsylvania)'
'ministers of the member states'


In [11]:
for m in ["instructed", "fewshot1", "fewshot3", "fewshot5", "structured", "cot"]:
    !python -m src.run_inference --method {m}

Loading microsoft/Phi-3-mini-4k-instruct ...
Loading weights: 100% 195/195 [00:28<00:00,  6.96it/s]
instructed: 100% 125/125 [05:05<00:00,  2.45s/it]
Generation took 5.1 min
{
  "n_total": 1000,
  "n_answerable": 493,
  "n_unanswerable": 507,
  "exact_match": 54.7,
  "f1": 61.99,
  "lenient_accuracy": 63.2,
  "has_ans_em": 65.11,
  "has_ans_f1": 79.9,
  "no_ans_accuracy": 44.58,
  "bleu": 62.42,
  "rouge_l": 80.77,
  "method": "instructed",
  "model": "microsoft/Phi-3-mini-4k-instruct",
  "generation_minutes": 5.1,
  "gpu": "Tesla T4",
  "max_new_tokens": 48
}

Saved results/instructed_preds.jsonl and results/instructed_scores.json
Loading microsoft/Phi-3-mini-4k-instruct ...
Loading weights: 100% 195/195 [00:20<00:00,  9.31it/s]
fewshot1: 100% 125/125 [09:28<00:00,  4.55s/it]
Generation took 9.5 min
{
  "n_total": 1000,
  "n_answerable": 493,
  "n_unanswerable": 507,
  "exact_match": 36.4,
  "f1": 47.07,
  "lenient_accuracy": 56.3,
  "has_ans_em": 49.29,
  "has_ans_f1": 70.93,
  "no_a

In [12]:
!python -m src.compare_results

Method      EM    F1     Lenient acc  Ans F1  Unans acc  BLEU   ROUGE-L  Minutes
----------  ----  -----  -----------  ------  ---------  -----  -------  -------
fewshot5    56.5  62.71  65.1         70.61   55.03      61.01  71.04    15.24  
structured  55.2  62.47  63.3         77.23   48.13      62.2   78.45    5.34   
instructed  54.7  61.99  63.2         79.9    44.58      62.42  80.77    5.1    
fewshot3    48.8  56.38  60.2         78.87   34.52      69.13  79.52    12.34  
cot         47.4  54.07  53.2         81.68   27.22      60.62  82.83    13.25  
fewshot1    36.4  47.07  56.3         70.93   23.87      57.5   71.57    9.47   
baseline    1.9   17.51  45.1         31.87   3.55       11.99  31.23    8.7    

Saved results/comparison.csv (sorted by F1)


In [13]:
!git config user.name "Elhamamy"
!git config user.email "aymenelhamamy@gmail.com"
!git commit --amend --reset-author --no-edit
!git push --force origin prompt-engineering

[prompt-engineering f0ef5e7] Add prompt engineering results
 24 files changed, 6327 insertions(+), 6 deletions(-)
 create mode 100644 results/comparison.csv
 create mode 100644 results/cot_preds.jsonl
 create mode 100644 results/cot_prompt_example.txt
 create mode 100644 results/cot_scores.json
 create mode 100644 results/fewshot1_preds.jsonl
 create mode 100644 results/fewshot1_prompt_example.txt
 create mode 100644 results/fewshot1_scores.json
 create mode 100644 results/fewshot3_preds.jsonl
 create mode 100644 results/fewshot3_prompt_example.txt
 create mode 100644 results/fewshot3_sample20_preds.jsonl
 create mode 100644 results/fewshot3_sample20_prompt_example.txt
 create mode 100644 results/fewshot3_sample20_scores.json
 create mode 100644 results/fewshot3_scores.json
 create mode 100644 results/fewshot5_preds.jsonl
 create mode 100644 results/fewshot5_prompt_example.txt
 create mode 100644 results/fewshot5_scores.json
 create mode 100644 results/instructed_preds.jsonl
 create mo